# Named Entity Recognition on Twitter

## CRF Model

#### Load and Inspect BTC Dataset (HuggingFace)

In [4]:
from datasets import load_dataset

# Load the Broad Twitter Corpus (BTC) dataset from HuggingFace
dataset = load_dataset("tner/btc")
tag_list = dataset['train'].features['tags'].feature.names

# Explore the available splits
print(dataset)

# Check tag list
print("\nTag list:", tag_list)

# Let's look at a few examples from the train split
for i in range(3):
    print(f"\nExample {i+1}")
    print("Tokens:", dataset['train'][i]['tokens'])
    print("Tags:  ", dataset['train'][i]['tags'])

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 6338
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 1001
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 2000
    })
})

Tag list: ['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']

Example 1
Tokens: ['I', 'hate', 'the', 'words', 'chunder', ',', 'vomit', 'and', 'puke', '.', 'BUUH', '.']
Tags:   [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6]

Example 2
Tokens: ['♥', '.', '.', ')', ')', '(', '♫', '.', '(', 'ړײ', ')', '♫', '.', '♥', '.', '«', '▓', '»', '♥', '.', '♫', '.', '.', '╝', '╚', '.', '.', '♫', 'Happy', 'New', 'Year']
Tags:   [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6]

Example 3
Tokens: ['Alesan', 'kenapa', 'mlm', 'kita', 'lbh', 'srg', 'galau', 'Poconggg', '\'"\'', 'TwitFAKTA', ':', 'Otak', 'lebih', 'aktif', 'di', 'malam', 'hari', 'dari', 'pada', 'di', 'pagi', 'hari'

In [5]:
# Convert numeric tags to strings
def sentence2labels(tag_ids):
    return [tag_list[tag_id] for tag_id in tag_ids]

#### Preprocess BTC for CRF

In [7]:
from datasets import load_dataset

# Load dataset and take a subset (optional, for speed during dev)
dataset = load_dataset("tner/btc")
btc_subset = dataset["train"].select(range(1000))  # adjust size as needed

# Feature extraction function for each token
def word2features(sent, i):
    word = sent[i]
    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],
        'word[-2:]': word[-2:],
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
    }
    if i > 0:
        prev_word = sent[i - 1]
        features.update({
            '-1:word.lower()': prev_word.lower(),
            '-1:word.istitle()': prev_word.istitle(),
            '-1:word.isupper()': prev_word.isupper(),
        })
    else:
        features['BOS'] = True  # Beginning of Sentence

    if i < len(sent) - 1:
        next_word = sent[i + 1]
        features.update({
            '+1:word.lower()': next_word.lower(),
            '+1:word.istitle()': next_word.istitle(),
            '+1:word.isupper()': next_word.isupper(),
        })
    else:
        features['EOS'] = True  # End of Sentence

    return features

# Convert the full sentence into a list of feature dicts and matching tags
def sentence2features(sentence_tokens):
    return [word2features(sentence_tokens, i) for i in range(len(sentence_tokens))]

def sentence2labels(tag_ids):
    return [tag_list[tag_id] for tag_id in tag_ids]

# Prepare the training data for CRF
X_train = [sentence2features(example["tokens"]) for example in btc_subset]
y_train = [sentence2labels(example["tags"]) for example in btc_subset]

# Check a sample
print("Sample features for first token of first sentence:")
print(X_train[0][0])
print("Corresponding label:", y_train[0][0])

Sample features for first token of first sentence:
{'bias': 1.0, 'word.lower()': 'i', 'word[-3:]': 'I', 'word[-2:]': 'I', 'word.isupper()': True, 'word.istitle()': True, 'word.isdigit()': False, 'BOS': True, '+1:word.lower()': 'hate', '+1:word.istitle()': False, '+1:word.isupper()': False}
Corresponding label: O


#### Train and Evaluate a CRF Model

In [9]:
import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.model_selection import train_test_split

# Split into train/test sets (80/20 split)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Initialize and train the CRF model
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,  # L1 regularization
    c2=0.1,  # L2 regularization
    max_iterations=100,
    all_possible_transitions=True
)

crf.fit(X_tr, y_tr)

# Predict on validation data
y_pred = crf.predict(X_val)

# Print flat classification report
labels = list(crf.classes_)
labels.remove('O')  # Optional: focus on named entities
print(metrics.flat_classification_report(y_val, y_pred, labels=labels, digits=3))

              precision    recall  f1-score   support

       B-ORG      0.286     0.049     0.083        41
       B-LOC      0.300     0.200     0.240        15
       I-LOC      0.000     0.000     0.000         7
       I-ORG      0.200     0.083     0.118        12
       B-PER      0.333     0.024     0.044        42
       I-PER      0.500     0.059     0.105        17

   micro avg      0.258     0.060     0.097       134
   macro avg      0.270     0.069     0.098       134
weighted avg      0.307     0.060     0.090       134



Key Observations:
* The recall is very low across the board, especially for I-LOC, I-ORG, and I-PER.

* The model is likely biased toward predicting "O" (non-entity), which is expected in NER on short tweets with sparse entities.

* This makes it a useful baseline to compare against more advanced models.

#### Error Analysis

In [12]:
def show_errors(X_val, y_val, y_pred, num_examples=5):
    error_count = 0

    for tokens, true, pred in zip(X_val, y_val, y_pred):
        # Check for any mismatches
        if true != pred:
            print("\n--- Mismatch Example ---")
            for i in range(len(tokens)):
                word = tokens[i]['word.lower()']
                gold = true[i]
                guess = pred[i]
                mark = "✔️" if gold == guess else "❌"
                print(f"{word:15} | True: {gold:7} | Pred: {guess:7} | {mark}")
            error_count += 1

            if error_count >= num_examples:
                break

# Show 5 error examples
show_errors(X_val, y_val, y_pred, num_examples=5)


--- Mismatch Example ---
new             | True: B-LOC   | Pred: O       | ❌
york            | True: I-LOC   | Pred: O       | ❌
i               | True: O       | Pred: O       | ✔️
love            | True: O       | Pred: O       | ✔️
ya              | True: O       | Pred: O       | ✔️

--- Mismatch Example ---
i               | True: O       | Pred: O       | ✔️
was             | True: O       | Pred: O       | ✔️
going           | True: O       | Pred: O       | ✔️
to              | True: O       | Pred: O       | ✔️
describe        | True: O       | Pred: O       | ✔️
#               | True: O       | Pred: O       | ✔️
earthakitt      | True: B-PER   | Pred: O       | ❌
as              | True: O       | Pred: O       | ✔️
an              | True: O       | Pred: O       | ✔️
accomplished    | True: O       | Pred: O       | ✔️
linguist        | True: O       | Pred: O       | ✔️
then            | True: O       | Pred: O       | ✔️
she             | True: O       | Pred: O       | 

Key Patterns from the Errors:
* False Negatives (missed entities):
    * The CRF consistently misses named entities, especially:

        * Locations (New York, Manchester Metropolitan Library)

        * People (Earthakitt, Ladotelli, Victor Valdes)

        * Organizations (KFC, Barcelona)

    * Many of these are proper nouns that appear in casual or Twitter-style syntax — e.g., lowercase, misspellings, hashtags.

* Wrong Label:
    * barcelona predicted as B-PER instead of B-ORG — a label confusion error

        * Likely caused by the model seeing barcelona used more often in person-like contexts in training.

## BiLSTM-CRF Model

#### Prepare the Data

In [16]:
import torch
from torch.nn.utils.rnn import pad_sequence
from collections import defaultdict

btc_subset = dataset["train"].select(range(5000))

# Build vocab and tag sets
word2idx = defaultdict(lambda: word2idx["<UNK>"])
tag2idx = {}
idx2tag = {}

# Create mappings
all_tokens = [word for sent in btc_subset['tokens'] for word in sent]
vocab = list(set(all_tokens))
word2idx = {word: i + 2 for i, word in enumerate(vocab)}
word2idx["<PAD>"] = 0
word2idx["<UNK>"] = 1

tag2idx = {tag: i for i, tag in enumerate(tag_list)}
idx2tag = {i: tag for tag, i in tag2idx.items()}

In [17]:
# Sequence Indexing + Padding
def encode_and_pad(tokens, tags, word2idx, tag2idx):
    token_ids = [word2idx.get(token, word2idx["<UNK>"]) for token in tokens]
    tag_ids = [tag2idx[tag] for tag in tags]
    return torch.tensor(token_ids), torch.tensor(tag_ids)

# Encode all examples
encoded = [encode_and_pad(ex["tokens"], [tag_list[i] for i in ex["tags"]], word2idx, tag2idx) for ex in btc_subset]

# Pad sequences
X_padded = pad_sequence([x for x, _ in encoded], batch_first=True, padding_value=word2idx["<PAD>"])
y_padded = pad_sequence([y for _, y in encoded], batch_first=True, padding_value=tag2idx["O"])  # pad to 'O'
mask = (X_padded != word2idx["<PAD>"])  # Create a mask for loss function

#### BiLSTM-CRF Model Definition

In [19]:
import torch.nn as nn
from torchcrf import CRF

class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim=100, hidden_dim=128, pad_idx=0):
        super(BiLSTM_CRF, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim // 2, num_layers=1, bidirectional=True, batch_first=True)
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)
        self.crf = CRF(tagset_size, batch_first=True)

    def forward(self, tokens, mask=None):
        embeddings = self.embedding(tokens)
        lstm_out, _ = self.lstm(embeddings)
        emissions = self.hidden2tag(lstm_out)
        return emissions

    def loss(self, tokens, tags, mask):
        emissions = self.forward(tokens, mask)
        loss = -self.crf(emissions, tags, mask=mask, reduction='mean')
        return loss

    def predict(self, tokens, mask):
        emissions = self.forward(tokens, mask)
        return self.crf.decode(emissions, mask=mask)

#### Train & Evaluate BiLSTM-CRF

In [21]:
# Model Init + Optimizer
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model params
vocab_size = len(word2idx)
tagset_size = len(tag2idx)
pad_idx = word2idx["<PAD>"]

# Initialize model
model = BiLSTM_CRF(vocab_size, tagset_size, pad_idx=pad_idx).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [22]:
# Split Train/Validation
from sklearn.model_selection import train_test_split

# Split into 80/20 for train/val
train_indices, val_indices = train_test_split(range(len(encoded)), test_size=0.2, random_state=42)
train_data = [encoded[i] for i in train_indices]
val_data = [encoded[i] for i in val_indices]

In [23]:
# Training Loop
from torch.nn.utils.rnn import pad_sequence
import random

def collate_fn(batch):
    tokens, tags = zip(*batch)
    tokens_pad = pad_sequence(tokens, batch_first=True, padding_value=word2idx["<PAD>"])
    tags_pad = pad_sequence(tags, batch_first=True, padding_value=tag2idx["O"])
    mask = (tokens_pad != word2idx["<PAD>"])
    return tokens_pad.to(device), tags_pad.to(device), mask.to(device)

# Training loop
EPOCHS = 15
BATCH_SIZE = 32

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    # Shuffle for each epoch
    random.shuffle(train_data)

    for i in range(0, len(train_data), BATCH_SIZE):
        batch = train_data[i:i+BATCH_SIZE]
        tokens_pad, tags_pad, mask = collate_fn(batch)

        loss = model.loss(tokens_pad, tags_pad, mask)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss:.4f}")

Epoch 1/15 - Loss: 1372.0836
Epoch 2/15 - Loss: 810.6617
Epoch 3/15 - Loss: 679.6377
Epoch 4/15 - Loss: 565.9646
Epoch 5/15 - Loss: 467.5545
Epoch 6/15 - Loss: 380.8275
Epoch 7/15 - Loss: 300.3295
Epoch 8/15 - Loss: 232.3852
Epoch 9/15 - Loss: 174.6644
Epoch 10/15 - Loss: 129.0898
Epoch 11/15 - Loss: 92.4220
Epoch 12/15 - Loss: 66.6200
Epoch 13/15 - Loss: 47.4489
Epoch 14/15 - Loss: 34.1563
Epoch 15/15 - Loss: 25.7032


In [24]:
# Evaluation on Validation Set
from seqeval.metrics import classification_report

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for i in range(0, len(val_data), BATCH_SIZE):
        batch = val_data[i:i+BATCH_SIZE]
        tokens_pad, tags_pad, mask = collate_fn(batch)
        predictions = model.predict(tokens_pad, mask)

        for pred_seq, true_seq, m in zip(predictions, tags_pad, mask):
            true_seq = true_seq[m].tolist()
            true_tags = [idx2tag[idx] for idx in true_seq]
            pred_tags = [idx2tag[idx] for idx in pred_seq]
            all_preds.append(pred_tags)
            all_labels.append(true_tags)

print(classification_report(all_labels, all_preds, digits=3, zero_division=0))

              precision    recall  f1-score   support

         LOC      0.447     0.437     0.442       341
         ORG      0.458     0.442     0.449       403
         PER      0.592     0.518     0.552       554

   micro avg      0.509     0.473     0.490      1298
   macro avg      0.499     0.466     0.481      1298
weighted avg      0.512     0.473     0.492      1298



### BERT

#### Load BTC Dataset & Subset It for BERT

In [27]:
from datasets import load_dataset

# Load BTC dataset
dataset = load_dataset("tner/btc")

# Work with ~2000 examples from training set
btc_small = dataset["train"].select(range(2000))

# Preview tags
tag_list = dataset['train'].features['tags'].feature.names
print("Tag List:", tag_list)

Tag List: ['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']


#### Setup BERT Components

In [29]:
from transformers import AutoTokenizer

# Use cased BERT to preserve capitalization (important for NER!)
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# Label to ID mapping
label2id = {label: i for i, label in enumerate(tag_list)}
id2label = {i: label for label, i in label2id.items()}

In [30]:
# Tokenize + Align Labels with Subwords
def tokenize_and_align_labels(example):
    tokenized = tokenizer(example["tokens"], truncation=True, is_split_into_words=True)
    word_ids = tokenized.word_ids()  # maps subword to word index

    aligned_labels = []
    previous_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            aligned_labels.append(-100)
        elif word_idx != previous_word_idx:
            aligned_labels.append(example["tags"][word_idx])  # label for first subword
        else:
            aligned_labels.append(-100)  # ignore subsequent subwords
        previous_word_idx = word_idx

    tokenized["labels"] = aligned_labels
    return tokenized


btc_tokenized = btc_small.map(tokenize_and_align_labels, batched=False)

#### Load BERT Model for Token Classification

In [32]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(tag_list),
    id2label=id2label,
    label2id=label2id
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


#### Create a DataLoader

In [34]:
from torch.utils.data import DataLoader
import torch

# Set format for PyTorch
btc_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# Create train/test split
from sklearn.model_selection import train_test_split

train_indices, val_indices = train_test_split(range(len(btc_tokenized)), test_size=0.2, random_state=42)
train_dataset = btc_tokenized.select(train_indices)
val_dataset = btc_tokenized.select(val_indices)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

#### Setup Trainer for BERT NER

In [36]:
import evaluate

metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = predictions.argmax(axis=-1)

    true_labels = [[id2label[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [id2label[p] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    # Optionally print detailed results
    print("\nDetailed entity scores:")
    for entity, score in results.items():
        if isinstance(score, dict):
            print(f"{entity}: {score}")

    # Return only scalar metrics to avoid warnings
    return {
        "overall_precision": results["overall_precision"],
        "overall_recall": results["overall_recall"],
        "overall_f1": results["overall_f1"],
        "overall_accuracy": results["overall_accuracy"],
    }

In [37]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="no",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=False,
)

In [38]:
from transformers import Trainer, DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [39]:
results = trainer.evaluate()
print(results)


Detailed entity scores:
LOC: {'precision': 0.0033142512805061767, 'recall': 0.18333333333333332, 'f1': 0.006510802012429714, 'number': 60}
ORG: {'precision': 0.009569377990430622, 'recall': 0.0449438202247191, 'f1': 0.015779092702169626, 'number': 89}
PER: {'precision': 0.009575923392612859, 'recall': 0.056451612903225805, 'f1': 0.016374269005847954, 'number': 124}
{'eval_loss': 1.9655460119247437, 'eval_model_preparation_time': 0.0111, 'eval_overall_precision': 0.004923903312444047, 'eval_overall_recall': 0.08058608058608059, 'eval_overall_f1': 0.009280742459396751, 'eval_overall_accuracy': 0.11954610835863833, 'eval_runtime': 32.8635, 'eval_samples_per_second': 12.172, 'eval_steps_per_second': 0.761}


### Final Model Comparison Table

| Model	| Precision	| Recall | F1 Score | Notes |
|--|--|--|--|--|
| CRF | 0.26 | 0.06 | 0.09 | Hand-crafted features, simple model
| BiLSTM-CRF | 0.058 | 0.032 | 0.041 | Learns from scratch, limited by lack of embeddings
| BERT (subset) | 0.011 | 0.216 | 0.021 | Pretrained contextual model, undertrained on small subset |

**Observation**:

* Even on a small subset, **BERT outperformed BiLSTM-CRF**, showing the advantage of using **pretrained embeddings**.

* **BiLSTM-CRF**, after increasing data and training epochs, finally achieved learnable performance (~4% F1), validating it as a fair baseline.

* **CRF** outperforms BiLSTM-CRF in this setup due to engineered features, highlighting how non-neural models can be effective on limited data.


### BERT on Full BTC Dataset

In [42]:
btc_full = dataset["train"]

btc_tokenized = btc_full.map(tokenize_and_align_labels, batched=False)
btc_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [43]:
# Pull one sample from the full BTC dataset
sample = dataset["train"][150] 

tokens = sample["tokens"]
tag_ids = sample["tags"]
tags = [tag_list[i] for i in tag_ids]

print("Tokens:", tokens)
print("Tags:  ", tags)


Tokens: ['#', 'Solar', 'confusion', 'sets', 'social', 'housing', 'projects', 'back', '|', 'Guardian', 'http://t.co/AEVSy5Kb']
Tags:   ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ORG', 'O']


In [44]:
from sklearn.model_selection import train_test_split

train_indices, val_indices = train_test_split(range(len(btc_tokenized)), test_size=0.2, random_state=42)
train_dataset = btc_tokenized.select(train_indices)
val_dataset = btc_tokenized.select(val_indices)

In [45]:
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time,Overall Precision,Overall Recall,Overall F1,Overall Accuracy
1,0.105700,0.129862,0.011100,0.606335,0.490842,0.542510,0.963721
2,0.089700,0.119950,0.011100,0.600000,0.615385,0.607595,0.966438
3,0.024700,0.128602,0.011100,0.622711,0.622711,0.622711,0.966757



Detailed entity scores:
LOC: {'precision': 0.49295774647887325, 'recall': 0.5833333333333334, 'f1': 0.534351145038168, 'number': 60}
ORG: {'precision': 0.5294117647058824, 'recall': 0.20224719101123595, 'f1': 0.2926829268292683, 'number': 89}
PER: {'precision': 0.6982758620689655, 'recall': 0.6532258064516129, 'f1': 0.675, 'number': 124}

Detailed entity scores:
LOC: {'precision': 0.6, 'recall': 0.7, 'f1': 0.6461538461538462, 'number': 60}
ORG: {'precision': 0.48717948717948717, 'recall': 0.42696629213483145, 'f1': 0.4550898203592814, 'number': 89}
PER: {'precision': 0.6666666666666666, 'recall': 0.7096774193548387, 'f1': 0.6875, 'number': 124}

Detailed entity scores:
LOC: {'precision': 0.6507936507936508, 'recall': 0.6833333333333333, 'f1': 0.6666666666666666, 'number': 60}
ORG: {'precision': 0.43820224719101125, 'recall': 0.43820224719101125, 'f1': 0.43820224719101125, 'number': 89}
PER: {'precision': 0.743801652892562, 'recall': 0.7258064516129032, 'f1': 0.7346938775510203, 'numbe

TrainOutput(global_step=300, training_loss=0.10755457282066345, metrics={'train_runtime': 1646.7549, 'train_samples_per_second': 2.915, 'train_steps_per_second': 0.182, 'total_flos': 133128549796704.0, 'train_loss': 0.10755457282066345, 'epoch': 3.0})

In [46]:
trainer.evaluate()


Detailed entity scores:
LOC: {'precision': 0.6507936507936508, 'recall': 0.6833333333333333, 'f1': 0.6666666666666666, 'number': 60}
ORG: {'precision': 0.43820224719101125, 'recall': 0.43820224719101125, 'f1': 0.43820224719101125, 'number': 89}
PER: {'precision': 0.743801652892562, 'recall': 0.7258064516129032, 'f1': 0.7346938775510203, 'number': 124}


{'eval_loss': 0.12860223650932312,
 'eval_model_preparation_time': 0.0111,
 'eval_overall_precision': 0.6227106227106227,
 'eval_overall_recall': 0.6227106227106227,
 'eval_overall_f1': 0.6227106227106227,
 'eval_overall_accuracy': 0.9667572319002717,
 'eval_runtime': 32.3925,
 'eval_samples_per_second': 12.349,
 'eval_steps_per_second': 0.772,
 'epoch': 3.0}

In [47]:
idx2label = {i: label for i, label in enumerate(tag_list)}

def show_bert_errors(model, dataset, tokenizer, max_examples=5):
    from torch.utils.data import DataLoader
    from transformers import DataCollatorForTokenClassification
    import torch

    model.eval()
    loader = DataLoader(dataset, batch_size=1, shuffle=False,
                        collate_fn=DataCollatorForTokenClassification(tokenizer))
    errors_shown = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"]
            attention_mask = batch["attention_mask"]
            labels = batch["labels"]

            predictions = model(input_ids, attention_mask=attention_mask).logits.argmax(-1)

            # Align tokens and filter out ignored tokens (-100)
            tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
            true = [idx2label[l.item()] for l, m in zip(labels[0], attention_mask[0]) if l != -100]
            pred = [idx2label[p.item()] for p, l in zip(predictions[0], labels[0]) if l != -100]

            if true != pred:
                print("\n--- Mismatch Example ---")
                for t, tr, pr in zip(tokens, true, pred):
                    mark = "✔️" if tr == pr else "❌"
                    print(f"{t:15} | True: {tr:7} | Pred: {pr:7} | {mark}")
                errors_shown += 1
                if errors_shown >= max_examples:
                    break


show_bert_errors(model, val_dataset, tokenizer)


--- Mismatch Example ---
[CLS]           | True: O       | Pred: O       | ✔️
UP              | True: O       | Pred: O       | ✔️
##DA            | True: B-ORG   | Pred: O       | ❌
##TE            | True: B-ORG   | Pred: B-ORG   | ✔️
##D             | True: O       | Pred: O       | ✔️
-               | True: O       | Pred: O       | ✔️
@               | True: O       | Pred: O       | ✔️
C               | True: O       | Pred: O       | ✔️
##ine           | True: O       | Pred: O       | ✔️
##plex          | True: O       | Pred: O       | ✔️
##M             | True: O       | Pred: O       | ✔️
##ov            | True: O       | Pred: O       | ✔️
##ies           | True: O       | Pred: O       | ✔️
hurt            | True: O       | Pred: O       | ✔️
by              | True: O       | Pred: O       | ✔️

--- Mismatch Example ---
[CLS]           | True: O       | Pred: O       | ✔️
.               | True: B-PER   | Pred: O       | ❌
@               | True: B-PER   | Pred: B-PER   |

#### BERT Error Analysis Summary
**Common Error Types**
1. **Subword alignment issues**

    * Example: `UP`, `##DA`, `##TE`, `##D` (→ “UPDATED”)

    * `B-ORG` label was only applied to part of the token (e.g., `##DA`), resulting in a mismatch.

    * Even though we use `-100` for ignored subwords, it still complicates predictions.

2. **False negatives (missed entities)**

    * Example: `Ivan##so##te` labeled `B-ORG`, but predicted as `O`.

    * These multi-token entities (split across subwords) are sometimes **not recognized at all**.

3. **False positives**

    * Example: `[CLS]` predicted as `B-ORG` — likely a rare fluke, but shows **BERT can be overconfident.**

4. **Entity boundary confusion**

    * Example: `marriage` should be `B-ORG`, but BERT labeled it `O`.

    * This may reflect **annotation ambiguity**, or the model not understanding context.

5. **Entity type confusion**

    * Example: `@` labeled `B-PER`, but predicted as `B-ORG`.

    * Common with **social media handles** where context is thin.